# Commande utiles shell

In [ ]:
# compter des lignes

wc -l test.txt

# chercher un mot

grep "rs123" fichier.txt

grep  -A 2 "rs123" fichier.txt #affiche 2 ligne après chaque match 

# Décompresser a la volée 

zcat chr11.vcf.gz | head

# Compter dans une colonne dans un fichier 

awk '$5 == 0.5' chr22_plink.frq | wc -l

# Type fichier

## VCF

un ficher **VCF** contient des variants génétiques, souvent : SNP, indels, génotypes pour plusieurs individus

exemple : 

```

##fileformat=VCFv4.2
##source=bcftools
#CHROM  POS     ID       REF ALT QUAL FILTER INFO FORMAT sample1 sample2
1       10583   rs1      G   A   29   PASS   .    GT     0/1     0/0
1       10611   rs2      C   G   45   PASS   .    GT     1/1     0/1

```

Les lignes commencent par # sont le header, 2 types :

- Métadonnées ##
- Lignes principales des colonnes #

### Les colonnes a connaitre : 

CHROM -> Chromosome

POS -> Position sur le chromosome

ID -> Nom du variant, souvent un rsID

REF -> Allèle de référence

ALT -> Allèle alternatif

QUAL -> Qualité du variant

FILTER -> Statut du filtre
    exemple : PASS = OK  |  autre chose = variant suspect ou filtré

INFO -> Infos complémentaires

FORMAT -> Indique ce que contiennent les colonnes échantillons


### Comprendre les génotypes :

0/0 -> homozygote référence

0/1 -> hétérozygote

1/1 -> homozygote alternatif

./. -> génotype manquant

### Ce qu’on filtre souvent dans un VCF

En pratique, on retire souvent :

- les variants trop rares selon l’objectif
- les variants avec trop de données manquantes
- les variants de mauvaise qualité
- les variants multialléliques selon le pipeline
- les indels si on veut un GWAS SNP simple

# Concept Génétique

## MAF

Fréquence de l'allèle minoritaire 

Par exemple : A = 95% G = 5%
Alors : MAF = 0.05

## Missingness

C'est le taux de données manquantes

Il y a 2 côtés : 
- le missingness par variant -> SNP mal génotypé dans bcp d'individus 
- le missingess par individu -> un individu avec bcp de SNPs manquants 

En pratique on filtre souvent : 
- variants : geno 0.05
- individus : mind 0.05

## HWE 

**Hardy-Weinberg Equilibrium**

La loi de Hardy-Weinberg stipule que les fréquences des allèles et des génotypes restent constantes dans une population idéale. 

Cette loi s’applique seulement si une population est grande, qu’il n’y a ni mutation, ni migration, ni sélection naturelle ou sexuelle. 

Les deux équations fondamentales sont p + q = 1 (pour les fréquences alléliques) et p² + 2pq + q² = 1 (pour les fréquences génotypiques).
En connaissant la fréquence d’un phénotype récessif (q²), il est possible de calculer les fréquences des autres allèles et génotypes. 
Le principe sert de référence pour les scientifiques afin de déterminer si une population est en cours d’évolution.

En savoir plus sur: https://jeretiens.net/le-principe-dequilibre-de-hardy-weinberg/


test statistique pour détecter des variants bizarres  avec : HWE p < 1e-6



## LD 

**Linkage Disequilibrium**  -> Situation dans laquelle deux gènes sont trouvés ensemble dans une population à une fréquence supérieure à celle prédite par le produit de leur fréquence individuelle

2 variant proche peuvent être corrélés et dans certains cas on veut évité d'avoir trop de SNPs redondants, donc on fait du LD pruning 

## Le génotype (GT)

Le génotype indique **quels allèles possède l'individu** a une postion 

```
CHROM POS      REF ALT GT
22    16050075 A   G   0|0
```

Signification des chiffres :

0 = REF

1 = ALT

2 = ALT2

3 = ALT3

Donc :

0|0 → A / A

0|1 → A / G

1|1 → G / G

Cas multi-allélique : REF = A ALT = G,T **alors** 0=A 1=G 2=T **donc** 1|2 = G/T 0|2 = A/T 2|2 = T/T


# bcftools 

## Lecture VCF

**Attention important** 

- view = filtrer / sélectionner
- query = extraire / formater
- head = voir seulement quelques lignes

In [ ]:
# voir header (-h --header-only)

bcftools view -h files.vcf.gz 
bcftools view -h files.vcf.g | less # car souvent trop long



# voir uniquement les variants (-H --no-header)

bcftools view -H files.vcf.gz 
bcftools view -H files.vcf.gz | head -5 # si que les 5 premiers 

# lister les samples (-l list-samples)

bcftools query -l files.vcf.gz 

# écrire la listet des sampes 

bcftools query -l chr22_portion.vcf.gz > liste_sample.txt

# extraire colonnes 

bcftools query -f 'FORMAT' file.vcf.gz 
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' files.vcf.gz

# compter les samples 

bcftools query -l files.vcf.gz | wc -l 

# compter les variants

bcftools view -H files.vcf.gz | wc -l 

# voir les chromosomes présents 

bcftools query -f '%CHROM\n' files.vcf.gz | sort | uniq 

## Filtrer un VCF   

**Attention**

- Avec view : header par défaut *donc* -H pour l’enlever
- Avec query : jamais de header *donc* -H inutile

**La commande -v**

- Cette commande accepte : snps, indels (ins et del), mnps, other
- Elle peut accepter plusieurs : -v TYPE1,TYPE2,TYPE3
- Si on veux exclure, à la place de filter : -V TYPE

*A savoir :* 

*- mnp (multi-nucleotide polymorphism) signifie que plusieurs bases changent mais même longueur*

*- other : variants complexes, multialléliques complexes, structural variants courts, combinaisons SNP + indel*

**La commande -o**

- sert à écrire dans un fichier sans bcftools écrit dans le terminal 

**La commande -O**

- sert à choisir le format de sortie 
- Ov  → VCF (texte)
- Oz  → VCF compressé (.vcf.gz)
- Ob  → BCF (binaire)
- Ou  → BCF non compressé

**Indexation**

Il existe 2 types d’index courants pour VCF.gz :

- .tbi -> ancien format, très courant
- .csi -> plus général, plus robuste pour de grons contigs/grandes coordonnées 

```bcftools index``` peut produire un .csi par défaut selon le contexte et la version.

**La commande -s**

Permet de filtrer les samples :
- -s → samples directement sur la ligne de commande
- -S → samples depuis un fichier texte




In [ ]:
# garder SNP uniquement 

bcftools view -v snps files.vcf.gz 

# garder SNP uniquement sans Headers 

bcftools view -H -v snps files.vcf.gz 

# Filtre SNP/extraire certaine colonnes/limite 3 lignes

bcftools view -v snps chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' | head -3 

# Compter les SNPs

bcftools view -H -v snps files.vcf.gz | wc -l

# Afficher les 3 premiers SNPs bialléliques 

bcftools view -m2 -M2 -v snps  chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' | head -3
    # -m2 : min 2 allèle (REF + 1ALT)
    # -M2 : max 2 allèle (donc 1 seul ALT)
    # donc on bialléliques uniquement 

# Compter le nombre allèle biallélique 

bcftools view -H -m2 -M2 -v snps  chr22.vcf.gz | wc -l

# Extraire une région génomique # -r CHR:START-END

bcftools view -r 22:16050000-16060000 chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' | head -3
    # avant d'utiliser -r, toujours vérifier le nom du chromosome : 22,chr22,Chr22,NC_000001.22
    # bcftools query -f '%CHROM\n' chr22.vcf.gz | head
    # ou bcftools view -h chr22.vcf.gz | grep contig

# Compter des variants dans une région

bcftools view -H -r 22:16050000-16060000 chr22.vcf.gz | wc -l

# Sauvegarder une région dans un VCF 

bcftools view -r 22:16050000-16060000 chr22.vcf.gz -Oz -o chr22_portion.vcf.gz 

# Indexer un VCF 

bcftools index chr22_portion.vcf.gz 
    # Indexé : filtrage rapide, accès par région, utilisation avec plink/bcftools 

# Filtre a partir des samples 

bcftools view -s SAMPLE file.vcf.gz
bcftools view -s SAMPLE1,SAMPLE2,SAMPLE3 file.vcf.gz
bcftools view -S liste_samples.txt file.vcf.gz

    # exemple d'utlisation 
    ## Afficher sur 3 variant le génotype pour un ou plusieurs samples 
    bcftools view -s HG00096 chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n'| head -3
    bcftools view -s HG00096,HG00097,HG00099 chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT\t]\n'| head -3 
                ### ne pas oublié après le GS \t ou mettre un espace pour espacer les génotype entre patient 

# PLINK

PLINK est un outil d'**analyse génétique** pour travailler sur bcp d'individus et bcp de SNPs 

Il est utilisé après un netoyage bcftools 

PLINK a été crée pour les analyses **GWAS** (association générique à grande échelle)

Il est :
- utilisé depuis 2007
- open source
- standard en génétique humaine
- utilisé dans publications scientifiques 
- utilisé en recherche + industrie 

Il sert : 
- calculer fréquences alléliques
- filtrer SNPs rares
- filtrer individus avec trop de missing
- PCA génétique (structure population)
- GWAS
- LD (linkage disequilibrium)
- convertir formats génétiques

Formats PLINK : *N'utilise pas directement VCF*

- .bed -> table binaire des génotypes biallélique 
- .bim -> information variant par varian (table descriptiondes variant)
- .fam -> table descriptives des samples 
- .log -> journal complet de la commande PLINK
- .nosex -> contient les individus dont le sexe est inconnu 

Pour il n'utilise pas les VCF car il n'est pas le fichier le plus optimiser pour faire des calculs rapides 

*A Savoir*

**GWAS** : Génome-Wide Association Study -> se qui veut dire tester chaque SNP pour voir s'il est associé a un trait 

**PCA** : Principal Component ANalysis -> résumer la variation génétique entre individus 



## Conversion VCF -> format PLINK 

In [ ]:
# Conversion VCF en format PLINK
plink --vcf input.vcf.gz --make-bed --out dataset
    ## --make-bed creer format blink binaire

#Conversion VCF en format PLINK avec un filtre MAF 

plink --vcf file.vcf.gz --maf 0.05 --make-bed --out dataset

# Conversion VCF en format PLINK avec ajout d'ID si absent 

plink --vcf file.vcf.gz --set-missing-var-ids template --make-bed --out dataset
    ## exemple template
    # @:# => chr:pos
    # @:#[b37]\$1,\$2 => chr:pos[b37]A1,A2   b37 = version du génome 
    # @:#\$1,\$2 => chr:posA1,A2

## MAF avec PLINK 

In [ ]:
# Commande PLINK pour MAF 

plink --bfile dataset --freq --allow-no-sex --out chr22_plink
    ## --allow-no-sex permet d'ignorer le sexe
    ## --out pour donnée un nom de sorti sinon plink. par default 

# Commande PLINK filtre MAF en creant un nouveau dataset 

link --bfile chr22_plink  --maf 0.05 --make-bed --out plink_filtered
        ## --make-bed écrit un nouveau jeu de fichiers BED/BIM/FAM

## Missing Data 

- Qu'est-ce que les données manquantes ?

exemple dans un genotype :

0/0

0/1

1/1

./. -> génotype manquant 

- Pourquoi c'est important ? 

Trop de missing entraine : fausse PCA, faux GWAS, biais statistiques **Donc on filtre**

- DEUX TYPES DE MISSING 

1) Missing par SNP = SNP avec trop de ./.

On filtre avec ```--geno 0.05``` => Supprime SNPs avec >5% missing 

2) Missing par individu = Individu avec trop de SNPs manquants 

On filtre avec ```--mind 0.05 ``` => Supprime individus avec >5% missing 


In [ ]:
# au moment de la conversion 

plink \
  --vcf chr22_S96_snps_biallelique.vcf.gz \
  --maf 0.05 \
  --make-bed \
  --allow-no-sex \
  --out chr22_S96_maf

# après la conversion 

plink \
  --bfile chr22_plink \
  --maf 0.05 \
  --make-bed \
  --allow-no-sex \
  --out chr22_plink_maf

    ## --make-bed écrit un nouveau jeu de fichiers BED/BIM/FAM

## --het

```--het``` sert à detecter des individus anormaux dans un dataset : 
- contamination ADN 
- mélange d'échantillons 
- consanguinité 
- erreur de génotypage
- outliers QC 

**Ce qu'il calcul** : 
- nombre homozygotes observé
- nombre homozygotes attendus 
- coefficient F (hétérozygotie)

**Formule** ```F = (O(HOM) - E(HOM)) / (N - E(HOM))```

**Interprétation** 
 
```
F ≈ 0   normal
F > 0   trop homozygote
F < 0   trop hétérozygote
```

**Quand l'utiliser** : 

Dans une pipeline QC GWAS : 

```
1. remove missing SNPs
2. remove missing individuals
3. filter MAF
4. LD prune
5. --het  ← ici
6. remove outliers
7. PCA
8. GWAS
```
**Conditions validation**

Il faut :

- beaucoup SNPs (≥ 50k recommandé)
- SNPs indépendants (LD pruning)
- autosomes seulement
- plusieurs individus

Sinon F devient instable.


In [ ]:
plink --bfile dataset --het --out dataset

sort -k6 -n dataset.het

## LD pruning

Dans le génome, bcp de SNPs sont corrélés (hérités ensemble)

Problème si on les garde : 
- PCA biaisé
- --het biaisé
- GWAS biaisé
- surpondération d'une région 

Donc on garde seulement des SNPs **indépendants**

PLINK utilisé une fenêtre glissante (genre 5 SNP) calcul le LD (corrélation r^2) entre SNPs, si r^2 supérieur au seuil => on enlève un SNP

| param  | signification       |Valeur Standard |
| ------ | ------------------- |----------------|
| window | taille fenêtre SNP  |      50        |
| step   | déplacement fenêtre |      5         |
| r2     | seuil corrélation   |      0.2       |


In [ ]:
plink --bfile dataset --indep-pairwise window step r2

Ce code produira deux fichiers :
- dataset.prune.in  => SNPs indépendants 
- dataset.prune.out => SNPs rétirés

Ensuite on récrée un jeu de donnée avec seulement les SNPs indéprendants 

In [ ]:
plink --bfile dataset --extract plink.prune.in --make-bed --out new_dataset

## --make-bed récrée un jeu de donnée 

Cette commande génère 2 fichiers :

- prefix.eigenvec
- prefix.eigenval


1) Fichier **eigenvec** :

*Format* => FID IID PC1 PC2 PC3 ...

*Exemple* =

```
HG00096 HG00096 -0.0165408 -0.0513594 -0.100299 -0.0559026 -0.0717423
HG00097 HG00097 0.194396 -0.224489 0.12821 0.423238 0.0102697
HG00099 HG00099 0.109652 -0.104717 0.224252 -0.04352 0.00328224
HG00100 HG00100 -0.0523295 -0.0503288 -0.0717088 -0.0761219 0.0418709
```

*Interprétation* :

- chaque individu = un point
- PC1 = axe principal
- PC2 = second axe

2) Fichier .eigenval

*Exemple* =

```
6.50
4.67
3.79
```

*Signification* = importance de chaque composante

Plus grand = plus de viriance expliqué 


## PCA

 1. Objectif de la PCA

La PCA (Principal Component Analysis) résume les différences génétiques entre individus.

Chaque individu est transformé en coordonnées génétiques :

PC1 PC2 PC3 PC4 PC5

Utilisation :

* détecter structure populationnelle
* détecter outliers
* QC avant GWAS
* corriger stratification population



2. Quand faire la PCA

Pipeline recommandé :
```
VCF
↓
Conversion PLINK
↓
QC (missing + MAF)
↓
LD pruning
↓
PCA
↓
Remove outliers
↓
GWAS
```


3. Commande PCA

```
plink --bfile dataset --pca 5 --out prefix
```

Exemple :

```
plink --bfile plink_100_prune --pca 5 --out plink_pca
```


4. Fichiers générés

```
prefix.eigenvec
prefix.eigenval
```

* .eigenvec → coordonnées individus
* .eigenval → importance des axes



5. Format eigenvec

```
FID IID PC1 PC2 PC3 PC4 PC5
```

Exemple :

```
HG00096 HG00096 -0.01 -0.05 0.02 ...
```

Chaque individu = un point dans l'espace génétique



6. Interprétation PCA

* Individus proches → génétiquement proches
* Individus éloignés → génétiquement différents
* Clusters → populations différentes



7. Détection d'outliers

On regarde plusieurs PCs :

* PC1
* PC2
* PC3

Pas seulement PC1.


 8. Colonnes PCA

```
col1 = FID
col2 = IID
col3 = PC1
col4 = PC2
col5 = PC3
col6 = PC4
col7 = PC5
```


9. Trouver outliers (tri)

PC1 :

```
sort -k3 -n plink_pca.eigenvec | head
sort -k3 -n plink_pca.eigenvec | tail
```

PC2 :

```
sort -k4 -n plink_pca.eigenvec | head
sort -k4 -n plink_pca.eigenvec | tail
```

PC3 :

```
sort -k5 -n plink_pca.eigenvec | head
sort -k5 -n plink_pca.eigenvec | tail
```


10. Identifier un outlier

Distribution normale :

```
-0.2  -0.1  0  0.1  0.2
```

Outlier :

```
-0.2  -0.1  0  0.1  0.2       0.8
                               ↑
```

Valeur isolée loin du reste → outlier



11. Après détection

Créer liste individus à retirer

Puis :

```
plink --bfile dataset --remove outliers.txt --make-bed --out dataset_clean
```


12. Résumé rapide

PCA = coordonnées génétiques individus
.eigenvec = positions individus
.eigenval = importance axes
regarder PC1 PC2 PC3
sort -k3 = PC1
sort -k4 = PC2
sort -k5 = PC3

Pipeline :

LD pruning → PCA → remove outliers → GWAS

## GWAS

1. Objectif

Le GWAS (Genome-Wide Association Study) teste l'association entre chaque SNP et un phénotype.

Structure générale :

Génotype (SNPs) vs Phénotype

Résultat :

* p-value
* odds ratio (OR)
* statistique du test


 2. Pipeline GWAS standard

```
VCF
↓
PLINK conversion
↓
QC (missing + MAF)
↓
LD pruning
↓
PCA
↓
remove outliers
↓
GWAS

```

3. Fichier phénotype

Format requis :

FID IID PHENO

Exemple :

HG00096 HG00096 1
HG00097 HG00097 2
HG00099 HG00099 1

Convention :

* 1 = contrôle
* 2 = cas
* -9 ou 0 = manquant



4. Covariables PCA

Utiliser fichier PCA :

plink_pca.eigenvec

Format :

FID IID PC1 PC2 PC3 PC4 PC5


5. Commande GWAS logistique

```
plink \
--bfile dataset \
--pheno pheno.txt \
--covar plink_pca.eigenvec \
--logistic \
--allow-no-sex \
--out gwas
```



6. Fichier de sortie

```
gwas.assoc.logistic
```

Colonnes principales :

```
CHR  SNP  BP  A1  TEST  NMISS  OR  STAT  P
```


7. Interprétation

* OR > 1 → risque
* OR < 1 → protecteur

P-value petite → association



8. Filtrer les vrais tests SNP

```
grep ADD gwas.assoc.logistic
```

ADD = test SNP

COV = covariables PCA



9. Trier par p-value

```
grep ADD gwas.assoc.logistic | grep -v NA | sort -k9 -g | head
```

Explication :

* -k9 = colonne p-value
* -g = numérique
* head = meilleurs SNPs



10. Seuil GWAS

Seuil classique :

5e-8

p < 5e-8 → significatif



11. Exemple interprétation

```
22:16283300 OR=0.21 P=0.0055
```

Interprétation :

* OR < 1 → effet protecteur
* p = 0.0055 → non significatif GWAS


 12. Résumé rapide

GWAS = SNP vs phénotype

--pheno = phénotype
--covar = PCA
--logistic = cas/contrôle
--linear = quantitatif

Résultat :

OR
p-value
SNP associé

Pipeline final :

QC → LD pruning → PCA → GWAS
